In [ ]:
%sql
SELECT concept_id, concept_name, standard_concept
FROM omop_vocab.concept
WHERE vocabulary_id = 'SNOMED' AND concept_code IN ('193570009','386805003','15188001');


we have redcap variable labled lbscat_a1c labled with Hemoglobin - g/dL and we mapped it to LOINC 4548-4
-lbscat_a1c,,Hemoglobin - g/dL,3004410,,Hemoglobin A1c/Hemoglobin.total in Blood,Measurement,LOINC,Lab Test,4548-4,S,skos:exactMatch,1,,,,

In [ ]:
%sql
-- 1. Distribution of lbscat_a1c values (concept 3000963), with HbA1c (3004410)
--    and hematocrit (3009542) for comparison
SELECT
  measurement_concept_id,
  CASE measurement_concept_id
    WHEN 3000963 THEN 'lbscat_a1c  (mapped: Hemoglobin g/dL)'
    WHEN 3004410 THEN 'import_hba1c (HbA1c %)'
    WHEN 3009542 THEN 'lbscat_hct  (Hematocrit %)'
  END                                                     AS source_field,
  COUNT(*)                                                AS n_rows,
  COUNT(DISTINCT person_id)                               AS n_persons,
  COUNT(value_as_number)                                  AS n_numeric,
  MIN(try_cast(value_as_number AS DOUBLE)) AS min_val,
  PERCENTILE_APPROX(try_cast(value_as_number AS DOUBLE), 0.05) AS p05,
  PERCENTILE_APPROX(try_cast(value_as_number AS DOUBLE), 0.50) AS median,
  PERCENTILE_APPROX(try_cast(value_as_number AS DOUBLE), 0.95) AS p95,
  MAX(try_cast(value_as_number AS DOUBLE)) AS max_val,
  ROUND(AVG(try_cast(value_as_number AS DOUBLE)), 2) AS mean_val,
  COLLECT_SET(unit_source_value)                          AS units_seen,
  COLLECT_SET(measurement_source_value)                   AS source_values_seen
FROM omop_survey.measurement
WHERE measurement_concept_id IN (3000963, 3004410, 3009542)
GROUP BY measurement_concept_id
ORDER BY measurement_concept_id;

In [ ]:
%sql
-- 2. How the lbscat_a1c values fall into the hemoglobin and HbA1c ranges
SELECT
  CASE
    WHEN try_cast(value_as_number AS DOUBLE) IS NULL            THEN 'null / non-numeric'
    WHEN try_cast(value_as_number AS DOUBLE) < 4                THEN '< 4 (implausible for both)'
    WHEN try_cast(value_as_number AS DOUBLE) < 10               THEN '4–<10  (HbA1c % range)'
    WHEN try_cast(value_as_number AS DOUBLE) < 11               THEN '10–<11 (overlap)'
    WHEN try_cast(value_as_number AS DOUBLE) <= 20              THEN '11–20  (Hemoglobin g/dL range)'
    ELSE                                         '> 20 (implausible for both)'
  END                         AS value_band,
  COUNT(*)                    AS n_rows
FROM omop_survey.measurement
WHERE measurement_concept_id = 3000963
GROUP BY 1
ORDER BY 1;

In [ ]:
%sql
-- 3. The strongest check: compare each person's lbscat_a1c value with their HbA1c (import_hba1c)
WITH lb AS (
  SELECT person_id, AVG(value_as_number) AS lbscat_a1c_val
  FROM omop_survey.measurement
  WHERE measurement_concept_id = 3000963 AND value_as_number IS NOT NULL
  GROUP BY person_id
),
a1c AS (
  SELECT person_id, AVG(try_cast(value_as_number AS DOUBLE)) AS hba1c_val
  FROM omop_survey.measurement
  WHERE measurement_concept_id = 3004410 AND value_as_number IS NOT NULL
  GROUP BY person_id
)
SELECT
  COUNT(*)                                              AS n_persons_with_both,
  ROUND(CORR(lb.lbscat_a1c_val, a1c.hba1c_val), 3)      AS correlation,
  ROUND(AVG(ABS(lb.lbscat_a1c_val - a1c.hba1c_val)), 2) AS mean_abs_difference
FROM lb JOIN a1c USING (person_id);

Data-quality follow-up (optional, not a mapping issue): the 13 values below 10 could be real anaemia, or someone may have typed the participant’s HbA1c into this field by mistake, given the field’s name. This query checks it by comparing those 13 values with each participant’s actual HbA1c:

In [ ]:
%sql
SELECT lb.person_id,
       lb.value_as_number  AS lbscat_a1c_val,
       a1c.value_as_number AS hba1c_val,
       hct.value_as_number AS hct_val
FROM omop_survey.measurement lb
LEFT JOIN omop_survey.measurement a1c
  ON a1c.person_id = lb.person_id AND a1c.measurement_concept_id = 3004410
LEFT JOIN omop_survey.measurement hct
  ON hct.person_id = lb.person_id AND hct.measurement_concept_id = 3009542
WHERE lb.measurement_concept_id = 3000963
  AND (try_cast(lb.value_as_number as double) < 10 OR try_cast(lb.value_as_number as double) > 20)
ORDER BY lb.value_as_number;

Even the outliers behave like hemoglobin: the 13 values below 10 g/dL all have matching low hematocrit (Hct/Hb ≈ 3.1–3.6, consistent with anemia), and the single value above 20 (21.1, with Hct 37) looks like a data-entry error for 12.1, which I’m flagging separately.

A short note for the data team:referring to the results in the cell7 above
A short note for the data team:

 